# 🏛️ LutherICPU — 100% Native 3D Neural Reconstructor & Simulation Model
### **Fully Automated Cloud GPU Pipeline with Hugging Face & GitHub Auto-Downloader**

This notebook runs **100% native LutherICPU code** from your repository with zero third-party model dependencies.
It automatically downloads the dataset from Hugging Face / mirrors, unzips it, executes 3D neural training from scratch, and downloads your ready-to-run 60 FPS simulation bundle.

---
### 🚀 **How to Run (1-Click):**
1. In the top menu, go to **Runtime** ➔ **Change runtime type** ➔ Select **T4 GPU**.
2. Click **Runtime** ➔ **Run all** (or press `Ctrl + F9`).
3. When finished, your trained `luther_simulation_bundle.zip` will automatically download to your computer.
4. On your PC, run `python simulate.py -b luther_simulation_bundle.zip` to launch the 60 FPS interactive 3D simulation!

In [ ]:
# Step 1: Environment Setup & Clone Your Native LutherICPU Repository
!nvidia-smi
!apt-get update -qq && apt-get install -y -qq colmap unzip wget
!pip install -q --upgrade pip
!pip install -q huggingface_hub plyfile tqdm scipy trimesh open3d onnxruntime opencv-python pillow

import os, shutil
if os.path.exists("/content/LutherICPU"):
    shutil.rmtree("/content/LutherICPU")

!git clone https://github.com/AryanParikh31/LutherICPU.git /content/LutherICPU
%cd /content/LutherICPU
if os.path.exists("requirements.txt"):
    !pip install -q -r requirements.txt

print("\n[SUCCESS] Native LutherICPU environment initialized successfully!")

In [ ]:
# Step 2: Automated Dataset Downloader (Hugging Face / Mirror with Auto-Unzip)
import os, sys, glob, shutil, zipfile
from huggingface_hub import snapshot_download

# Scene Selection (Options: 'drjohnson', 'playroom', 'truck', 'train')
SCENE_NAME = "drjohnson"  # @param ["drjohnson", "playroom", "truck", "train"]

workspace_dir = "/content/dataset"
images_target = os.path.join(workspace_dir, "images")
sparse_target = os.path.join(workspace_dir, "sparse", "0")

if os.path.exists(workspace_dir):
    shutil.rmtree(workspace_dir)
os.makedirs(images_target, exist_ok=True)
os.makedirs(sparse_target, exist_ok=True)

print(f"[*] Automatically downloading '{SCENE_NAME}' dataset from Hugging Face...")
try:
    # Fast download from Hugging Face
    pattern = f"db/{SCENE_NAME}/*" if SCENE_NAME in ["drjohnson", "playroom"] else f"tandt/{SCENE_NAME}/*"
    hf_dir = snapshot_download(
        repo_id="alexmkwizu/gaussian_training_datasets",
        repo_type="dataset",
        allow_patterns=[pattern],
        local_dir="/content/hf_dataset"
    )
    
    # Locate downloaded scene folder
    scene_root = None
    for root, dirs, files in os.walk("/content/hf_dataset"):
        if os.path.basename(root).lower() == SCENE_NAME.lower():
            scene_root = root
            break
            
    if scene_root:
        # Copy images
        src_img_dir = os.path.join(scene_root, "images")
        if os.path.exists(src_img_dir):
            for f in os.listdir(src_img_dir):
                if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                    shutil.copy2(os.path.join(src_img_dir, f), os.path.join(images_target, f))
        
        # Copy sparse calibration if available
        src_sparse_dir = os.path.join(scene_root, "sparse", "0")
        if os.path.exists(src_sparse_dir):
            for f in os.listdir(src_sparse_dir):
                shutil.copy2(os.path.join(src_sparse_dir, f), os.path.join(sparse_target, f))

except Exception as e:
    print(f"[!] Hugging Face download note: {e}. Falling back to INRIA direct archive mirror...")
    !wget -q --show-progress -O /content/tandt_db.zip https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/datasets/input/tandt_db.zip
    with zipfile.ZipFile("/content/tandt_db.zip", 'r') as zip_ref:
        zip_ref.extractall("/content/tandt_db_extracted")
    
    # Find matching folder
    for root, dirs, files in os.walk("/content/tandt_db_extracted"):
        if os.path.basename(root).lower() == SCENE_NAME.lower():
            src_img = os.path.join(root, "images")
            if os.path.exists(src_img):
                for f in os.listdir(src_img):
                    if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                        shutil.copy2(os.path.join(src_img, f), os.path.join(images_target, f))
            src_sp = os.path.join(root, "sparse", "0")
            if os.path.exists(src_sp):
                for f in os.listdir(src_sp):
                    shutil.copy2(os.path.join(src_sp, f), os.path.join(sparse_target, f))
            break

num_imgs = len(os.listdir(images_target))
print(f"\n[SUCCESS] Loaded {num_imgs} photographs for scene '{SCENE_NAME}' into native workspace.")
print(f"Images location: {images_target}")

In [ ]:
# Step 3: Run Native Luther Camera Ingestion & Calibration
%cd /content/LutherICPU
!python -m luther_core.ingestion --images /content/dataset/images --output /content/scene_output

In [ ]:
# Step 4: Train Native Luther 3D Model from Scratch (30,000 Iterations Optimization)
%cd /content/LutherICPU
from luther_pipeline.simulation_model import LutherICPU

colmap_sparse_dir = "/content/dataset/sparse/0" if os.path.exists("/content/dataset/sparse/0/cameras.bin") else None

print("=" * 80)
print(f"[*] Training Native LutherICPU 3D Photometric Model for '{SCENE_NAME}' from Scratch...")
print("=" * 80)

model = LutherICPU(output_dir="/content/scene_output/model", max_ram_gb=14.0)
results = model.simulate(
    images_path="/content/dataset/images",
    colmap_path=colmap_sparse_dir,
    iterations=30000,
    scene_name=SCENE_NAME,
    texture_resolution=4096,
    render_simulation=True
)

print("\n" + "=" * 80)
print("[SUCCESS] Native LutherICPU 3D Model Training Complete!")
print(f"Primitives / Splats: {results.get('num_dense_points', 0):,}")
print(f"OBJ Model:           {results.get('obj_path')}")
print(f"4K Diffuse Atlas:    {results.get('diffuse_png_path')}")
print(f"Binary Mesh:         {results.get('binary_mesh_path')}")
print("=" * 80)

In [ ]:
# Step 5: Package Native 3D Simulation Bundle & Auto-Download to Local PC
import os, zipfile
from google.colab import files

out_model_dir = "/content/scene_output/model"
bundle_zip = f"/content/luther_simulation_{SCENE_NAME}_bundle.zip"

print("[*] Packaging Native Luther 3D Simulation Assets...")
with zipfile.ZipFile(bundle_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, fnames in os.walk(out_model_dir):
        for f in fnames:
            full_p = os.path.join(root, f)
            rel_p = os.path.relpath(full_p, out_model_dir)
            zipf.write(full_p, rel_p)

size_mb = os.path.getsize(bundle_zip) / (1024 * 1024)
print(f"\n[SUCCESS] Simulation Bundle Created: {size_mb:.1f} MB")
print("[*] Initiating automatic download to your computer...")
files.download(bundle_zip)
print("\n[NEXT STEP] On your local PC, run:")
print(f"python simulate.py -b luther_simulation_{SCENE_NAME}_bundle.zip")